<u>#**Analysis of Polymarket weather markets**</u>

#Focusing on does the crowd favorite win?

First we need to obtain the data we want to work with for this analysis. To get the data there are three main API's that are free from polymarket, these are the GAMMA API, COLB API and DATA API. GAMMA API provides answers to questions like if a particular market exists on the site. COLB API provides info regarding the order books for markets and DATA API provides info about each user interacting with a market. 


In [8]:
import requests
import pandas as pd
import numpy as np 
import json
import time 
from datetime import datetime, timezone, timedelta


def get_historic_events(days = 365):
    """This function pulls all event data regarding the events returning 2 data frames, the event df and the markets for a event """

    #create date range for scrapping
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days)

    offset = 0
    events = []

    while True:
    
        time.sleep(10)
        #filiters to get the info we want 
        params = {
            "tag_slug" : "weather", #only bring weather events
            "closed" : True, #event had ended 
            "limit" : 100, #requests a max of 100 events 
            "offset" : offset, #how far to move down
            "ascending" : False, #sort by newest to oldest
            "order" : end_date, #sort by the ending date 
            "end_date_min" : start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end_date_max" : end_date.strftime("%Y-%m-%dT%H:%M:%SZ") #create event range
           
            }

        events_response = requests.get(
            "https://gamma-api.polymarket.com/events",
            params=params)

        events_response.raise_for_status()

        #converts polymarket data into a dict
        batch = events_response.json()

        #extend not append to not get ladder of lists
        events.extend(batch)

        print(f"fetched: {len(events)} events")

        if len(batch) < 100:
            break
        else:
            offset += len(batch)


    #events are repeated markets ie some bet
    
    markets_rows = []
    events_clean = []

    for e in events:
        markets = e.pop("markets", []) or []
        events_clean.append(e)

        for m in markets:
            m["event_id"] = e.get("id")
            m["event_title"] = e.get("title")
            m["event_slug"] = e.get("slug")
            markets_rows.append(m)

    df_events = pd.DataFrame(events_clean)
    df_markets = pd.DataFrame(markets_rows)

    print(f"Total markets: {len(df_markets)}")

    return df_events, df_markets


df_events, df_markets = get_historic_events(days = 20)


fetched: 100 events
fetched: 200 events
fetched: 300 events
fetched: 400 events
fetched: 500 events
fetched: 600 events
fetched: 700 events
fetched: 800 events
fetched: 900 events
fetched: 1000 events
fetched: 1100 events
fetched: 1200 events
fetched: 1300 events
fetched: 1400 events
fetched: 1500 events
fetched: 1600 events
fetched: 1700 events
fetched: 1800 events
fetched: 1900 events
fetched: 1923 events
Total markets: 21288


In [ ]:

#convert dates in 2026-09-05T12:00:00Z to unix format
def to_unix_timestamp(date_text):
    date = datetime.fromisoformat(
        date_text.replace("Z", "+00:00")
    )
    return int(date.timestamp())

In [ ]:
def price_history(clobtoken_id, start_ts, end_ts, fidelity):
    """This function gets all the market pricing history for a single market
    clobtoken_id is the clob token id 
    start_ts is is time range start for price history 
    end_ts is time range end for price history
    fidelity is the resolution we want to collect the data at
    """


    parms = {
        "market" : clobtoken_id,
        "startTs" : to_unix_timestamp(start_ts),
        "endTs": to_unix_timestamp(end_ts),
        "interval": "all",
        "fidelity": fidelity,
    }

    response = requests.get("https://clob.polymarket.com/prices-history", parms)
    response.raise_for_status()

    price_history = pd.DataFrame(response.json().get("history", []))

    if price_history.empty:
        return price_history
    else:
        price_history["timestamp"] = pd.to_datetime(price_history["t"], unit="s", utc=True)
        price_history["price"] = price_history["p"]
        return price_history[["timestamp", "price"]].sort_values("timestamp")



We can now collect data about the weather events and the markets price history. However we now need to proccess and clean the data to only have "temperature prediction" markets that we want to investigate. 

In [38]:

import re
import matplotlib.pyplot as plt

print(df_events["title"])

def event_classifier(title_column):
    """create set of defined rules that classify each event """

    #various regex rules to identify the type of event 
    EVENT_TYPE_RULES = [
    ('Temperature', r'\b(?:temperatures?|hottest|coldest|heat\s*waves?|degrees?\s*[cf])\b'),
    ('Earthquake', r'\b(?:earthquakes?|seismic|tsunamis?)\b'),
    ('Hurricane / Typhoon', r'\b(?:hurricanes?|typhoons?|cyclones?|tropical\s+(?:storms?|depressions?))\b'),
    ('Solar / Space Wx', r'\b(?:solar|geomagnetic|auroras?|space\s+weather|coronal)\b'),
    ('Flu / Respiratory', r'\b(?:flu|influenza|respiratory|rsv|covid(?:\s*19)?|h5n1|bird\s+flu)\b'),
    ('Infectious Disease', r'\b(?:measles|mpox|monkeypox|ebola|dengue|malaria|polio|outbreaks?|pandemics?|infectious\s+disease)\b'),
    ('Snow / Ice', r'\b(?:snow\w*|blizzards?|ice|icy|freez\w*|frost|hail\w*)\b'),
    ('Tornado', r'\b(?:tornado(?:es|s)?|twisters?)\b'),
    ('Rain / Flood', r'\b(?:rain\w*|flood\w*|precipitation|monsoons?)\b'),
    ('General Storm', r'\b(?:storms?|thunderstorms?|wind\w*)\b'),
    ('Drought', r'\b(?:droughts?)\b'),
    ('Wildfire', r'\b(?:wildfires?|forest\s+fires?|bushfires?)\b'),
    ('Volcano', r'\b(?:volcan\w*|eruptions?)\b'),
]

    classification = []

    for title in title_column:
        classified = "other"

        for pattern in EVENT_TYPE_RULES:
            if re.search(pattern[1], title, flags=re.IGNORECASE):
                classified = pattern[0]
                break          

        classification.append(classified)
    df_events["event_type"] = classification

event_classifier(df_events["title"])

0       Will Oregon reach D4 (Exceptional Drought) by ...
1              Cyclosporiasis cases in U.S. by August 31?
2            Highest Mt. Washington wind speed in August?
3                         Precipitation in NYC in August?
4                   Precipitation in Hong Kong in August?
                              ...                        
1918       Highest temperature in Toronto on September 5?
1919    Highest temperature in Mexico City on Septembe...
1920         Highest temperature in Jinan on September 5?
1921     Highest temperature in Zhengzhou on September 5?
1922       Which cities face tornado risk on September 4?
Name: title, Length: 1923, dtype: object
